In [ ]:
instruction_data_path = "../data/deposit-account-agreement.jsonl"
from datasets import load_dataset

In [ ]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)
print(instruction_dataset)
print(instruction_dataset[0])

In [ ]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("Prompt", "")).strip()
    input_text = str(record.get("Question", "")).strip()
    output_text = str(record.get("Answer", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [ ]:
instruction_dataset = instruction_dataset.map(format_instruction_record)


In [ ]:
instruction_dataset


In [ ]:
print(instruction_dataset[5]["text"])

In [ ]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

In [ ]:
#!uv pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "../source/deposit-account-agreement.pdf"

    # Base causal language model that we will fine-tune on customer account-agreement text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "../output/daa_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "../adapter/daa_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "../output/daa_tinyllama_lora_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 10.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [ ]:
config = Config()

In [ ]:
config

In [ ]:
import json
print(json.dumps(asdict(config), indent=2))

In [ ]:
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [ ]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

pdf_pages = extract_pdf_pages(config.pdf_path)
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")



In [ ]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

cleaned_pages = []

In [ ]:
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print("Total cleaned pages:", len(cleaned_pages))

In [ ]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [ ]:
paragraph_records = split_into_paragraph_records(cleaned_pages)
print("Total paragraph records:", len(paragraph_records))

In [ ]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

In [ ]:

# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "daa_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

In [ ]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================
from datasets import Dataset
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)

In [ ]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

In [ ]:
import os
from datasets import load_dataset
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
print("HF_TOKEN loaded from .env:", api_key)

# 1. Load your local or existing dataset
# (Replace this with how you currently load your dataset object)
#dataset = load_dataset("json", data_files="your_data.json") 

# 2. Push directly using your specific username and Write token
# This bypasses all .env caching issues entirely
text_dataset.push_to_hub(
    repo_id="Nashxi/daa-non-instruction-DS",  # Must include username/
    token=api_key       # Paste the raw token string
)

In [ ]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

In [ ]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

In [ ]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets['train']['input_ids'][0]

In [ ]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [ ]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

In [ ]:
sample = final_dataset["train"][0]
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

In [ ]:
# ============================================================
# 13. Load base model
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

In [ ]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [ ]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

from transformers import TrainingArguments

In [ ]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [ ]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

In [ ]:
for log in trainer.state.log_history:
    print(log)

In [ ]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

In [ ]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

In [ ]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================
repo_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-live"

In [ ]:
import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
print("HF_TOKEN loaded from .env:", api_key)

model.push_to_hub(
    repo_id, token=api_key,
    private=True
)

In [ ]:
tokenizer.push_to_hub(
    repo_id,
    private=True
)

In [ ]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
# yo bro - start again with a fresh model and tokenizer load, then apply the LoRA adapter.

from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [ ]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

In [ ]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [ ]:
inference_model.eval()

In [ ]:
print("Base model + LoRA adapter loaded successfully for inference.")

In [ ]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Uniform Transfers to Minors Act/Uniform Gifts to Minors Act (UTMA/UGMA) account",
    "When survivorship rights apply Except as otherwise stated in this paragraph",
    "What we do How does Chase protect my personal information?",
    "If a joint account has rights of survivorship, and one joint owner dies",
]

prompts1 = [

    "If one owner of a marital account dies, the survivor is ",
]


In [ ]:
#now test this with the generate_completion function
import textwrap
for prompt in prompts1:
    print("=" * 80)
    print(f"Prompt:\n{prompt}\n")
    completion = generate_completion(prompt, max_new_tokens=120)
    print(f"Completion:\n{completion}\n")
    # Wrap and print directly
    print(textwrap.fill(completion, width=80))


In [ ]:
# note that till now, we have only base model and Lora Adapter
# now i will merge them into a single model

# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "../output/daa_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

offload_model_dir = "../output/daa_tinyllama_offload_model"
os.makedirs(offload_model_dir, exist_ok=True)


In [ ]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=offload_model_dir  # Add this line
)


In [ ]:
# Print the structure to find the true path of the lm_head
print(base_model) 

# Or inspect the state dict keys directly
#print([k for k in base_model.state_dict().keys() if "lm_head" in k])

# Force-feed the script the exact dictionary key string it wants
state_dict = base_model.state_dict()
if "lm_head" in state_dict:
    state_dict["base_model.model.model.lm_head"] = state_dict["lm_head"]

# Or inspect the state dict keys directly
#print([k for k in base_model.state_dict().keys() if "lm_head" in k])

In [ ]:
import os
import torch
from peft import PeftModel
from safetensors.torch import load_file

sf_path = os.path.join(config.adapter_dir, "adapter_model.safetensors")
bin_path = os.path.join(config.adapter_dir, "adapter_model.bin")

if os.path.exists(sf_path):
    adapter_weights = load_file(sf_path)
else:
    adapter_weights = torch.load(bin_path, map_location="cpu")

# 3. Apply the patch: Remap the flat key to the nested one PEFT wants
if "lm_head" in adapter_weights:
    adapter_weights["base_model.model.model.lm_head"] = adapter_weights.pop("lm_head")
if "lm_head.weight" in adapter_weights:
    adapter_weights["base_model.model.model.lm_head.weight"] = adapter_weights.pop("lm_head.weight")



In [ ]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)
model_with_adapter.load_state_dict(adapter_weights, strict=False)

In [ ]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [ ]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

In [ ]:
import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# ============================================================
# 20. Push LoRA adapter + base model Merged model to Hugging Face Hub
# ============================================================
repo_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-merged"

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
print("HF_TOKEN loaded from .env:", api_key)

merged_model.push_to_hub(
    repo_id, token=api_key,
    private=True
)

# Stage 2: Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model
 
In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw pharma PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured pharma instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw pharma text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on pharma Q&A data
   ↓
Final instruction-tuned pharma LoRA adapter
```

This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

What changes in instruction fine-tuning?

For non-instruction fine-tuning, the data looked like raw text:

```text
Metformin is one of the most widely prescribed oral antihyperglycemic agents...
```

For instruction fine-tuning, the data looks like:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
```

This teaches the model not only deposit account agreement language, but also how to answer user instructions.

In [ ]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"